In [3]:
# ============================================================
# 032_monitor_vc_daily
# ============================================================
#
# Overview
# ----------------
# Daily VC monitoring notebook that:
#   1) loads configured VC Monitoring Targets (from Notion),
#   2) fetches updates (RSS/HTML + optional NewsAPI),
#   3) normalizes items into Event candidates (date-safe),
#   4) applies freshness + optional window filtering,
#   5) deduplicates strictly (in-memory + Notion),
#   6) writes NEW Events to Notion via 029 wrappers only,
#   7) produces a clean Daily Run Summary (Markdown + Slack text).
#
# This is a Day30+ downstream notebook in a layered architecture:
# - 028_config_and_state.ipynb provides execution context:
#     config, scanner_config, run_id, run_stats, logger, state helpers
# - 029_notion_clients_and_io.ipynb provides all Notion I/O wrappers:
#     list/query/create/update wrappers (no raw Notion calls here)
# This notebook implements ONLY domain logic for VC monitoring.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - VC Monitoring Targets from Notion (via 029 wrappers)
#   - scanner_config (window_start/window_end, enable_window_filter, freshness_days, debug flags)
#   - state (optional): last_vc_scan_at, prior run counters
#
# Outputs:
#   - New Events written to Notion Events DB (via 029 wrappers)
#   - Updated state (optional): last_vc_scan_at, counters (created/skipped/failed)
#   - Daily Run Summary: Markdown + Slack snippet text
#   - Optional: local summary .md file
#
# Structure
# ----------------
# Cell 01: Imports and Safe Defensive Handles
# Cell 02: Config (freshness/window/debug) + Derived Cutoffs
# Cell 03: Load VC Monitoring Targets via 029
# Cell 04: Fetch Helpers (RSS first, HTML fallback, optional NewsAPI)
# Cell 05: Normalize Items -> Event Candidates (date required, date-safe)
# Cell 06: Dedup + Write Events via 029 (idempotent rerun-safe)
# Cell 07: State Update + Daily Run Summary (Markdown + Slack)
# Cell 08: Optional Local Summary Save
#
# Key Rules / Guarantees
# ----------------
# 1) No Notion writes before Cell 06.
# 2) Dedup is strict:
#    - in-memory dedup by dedup_key (within run)
#    - Notion dedup by querying dedup_key (across runs)
#    - rerunning must not create duplicates.
# 3) Date strategy (critical):
#    - If published date is unavailable and cannot be extracted from title/summary, DROP the item.
#    - Event date is normalized to YYYY-MM-DD before Notion writes.
# 4) Filters:
#    - Freshness filter always applies (freshness_cutoff_dt).
#    - Window filter is optional: enable_window_filter (default True).
#      When False, only freshness filter applies (useful for backfills/tests).
# 5) Robust stats (run_stats):
#    - All counters must be initialized with setdefault before increment (avoid KeyError).
#    - targets_processed is incremented once per target fetch attempt (not per item).
# 6) Per-target isolation:
#    - one target failure must not stop the whole run.
#
# Notes
# ----------------
# - Depends strictly on 028 (config/state/logger) and 029 (Notion I/O).
# - Does NOT implement infra, Notion clients, retries, or raw HTTP.
# - All Notion side effects go through 029 wrappers only.
# - Deterministic dedup_key computed from normalized URL + title + date + source label (stable).
# - Debug logging is bounded (max N samples) to avoid noisy runs.
#

In [16]:
# ============================================================
# Cell 01 — Imports and defensive bootstrap (exec 028/029)
# ============================================================
# Overview:
#   Execute 028_config_and_state.ipynb and 029_notion_clients_and_io.ipynb in THIS kernel
#   and defensively bind runtime objects (run_id/logger/config/state helpers, Notion wrappers).
#   This cell MUST NOT load env vars or initialize clients (owned by 028/029).
#
# Inputs / Outputs:
#   Inputs:  028_config_and_state.ipynb, 029_notion_clients_and_io.ipynb
#   Outputs: run_id, logger, config, get_state, update_state (+ optional Notion wrappers)
#
# Notes:
#   - No env loading here (028 owns env/secrets)
#   - No Notion client init here (029 owns Notion side effects)
#   - Use %run -i to inject into current globals; then bind defensively

from __future__ import annotations

import os
import json
from pathlib import Path
from datetime import datetime, date
from typing import Any, Dict, Optional

from datetime import date, timedelta

def compute_next_check(cadence: str, base: date) -> date | None:
    c = (cadence or "").upper()
    if c == "DAILY":
        return base + timedelta(days=1)
    if c == "WEEKLY":
        return base + timedelta(days=7)
    if c == "BIWEEKLY":
        return base + timedelta(days=14)
    if c == "MONTHLY":
        return base + timedelta(days=30)  # 最初は雑でOK
    return None
    
# ------------------------------------------------------------
# 1) Execute 028 and 029 (inject into this kernel)
# ------------------------------------------------------------
HERE = Path.cwd()
nb028 = HERE / "028_config_and_state.ipynb"
nb029 = HERE / "029_notion_clients_and_io.ipynb"

if not nb028.exists():
    raise FileNotFoundError(f"Cannot find: {nb028}")
if not nb029.exists():
    raise FileNotFoundError(f"Cannot find: {nb029}")

get_ipython().run_line_magic("run", f"-i {nb028}")
get_ipython().run_line_magic("run", f"-i {nb029}")

print("✅ 028 and 029 executed in current kernel")

# ------------------------------------------------------------
# 2) Defensive binding of execution context from 028
# ------------------------------------------------------------
def _noop_logger():
    class _L:
        def info(self, *a, **k): print(*a)
        def warning(self, *a, **k): print(*a)
        def error(self, *a, **k): print(*a)
        def debug(self, *a, **k): pass
    return _L()

run_id = globals().get("run_id") or f"run_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
logger = globals().get("logger") or _noop_logger()

config = (
    globals().get("config")
    or globals().get("CONFIG")
    or globals().get("cfg")
    or globals().get("settings")
    or {}
)

get_state = globals().get("get_state")
update_state = globals().get("update_state")

def _fallback_get_state(key: str, default=None):
    return default

def _fallback_update_state(key: str, value):
    return None

if not callable(get_state):
    get_state = _fallback_get_state
if not callable(update_state):
    update_state = _fallback_update_state

# ------------------------------------------------------------
# 3) Paths (optional, based on config; safe defaults)
# ------------------------------------------------------------
paths_cfg = config.get("paths", {}) if isinstance(config, dict) else {}
download_dir = Path(paths_cfg.get("download_dir", "data/downloads"))
artifacts_dir = Path(paths_cfg.get("artifacts_dir", "artifacts"))

# For monitor notebooks you might not need these, but harmless to prepare:
for d in [download_dir, artifacts_dir]:
    d.mkdir(parents=True, exist_ok=True)

today_str = date.today().isoformat()

logger.info(f"[{run_id}] Cell 01 ready")
logger.info(f"[{run_id}] today={today_str}")

# ------------------------------------------------------------
# 4) Defensive binding of Notion wrappers from 029 (do NOT require all)
# ------------------------------------------------------------
# Prefer generic wrappers if they exist; don't hard-fail here.
query_monitoring_targets = globals().get("query_monitoring_targets")
create_event = globals().get("create_event") or globals().get("create_event_inbox") or globals().get("write_event_to_notion")
find_duplicate_event = globals().get("find_duplicate_event") or globals().get("check_event_exists")

logger.debug(
    f"[{run_id}] wrappers: "
    f"query_monitoring_targets={callable(query_monitoring_targets)}, "
    f"create_event={callable(create_event)}, "
    f"find_duplicate_event={callable(find_duplicate_event)}"
)


✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           8fc0a521-19f9-49be-a7d9-2584732c4be7
  - Execution start:  2026-01-29T00:57:57.843107+00:00
  - Run date:         2026-01-29
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-22 to 2026-01-29)
    - tasks       : 14 days (2026-01-15 

In [17]:
# ============================================================
# Cell 02 — Config time window & scanner defaults (VC)
# ============================================================
# Overview:
#   Resolve VC scan window and scanner defaults from config (028-owned),
#   preferring 028 time helpers when available. Load last_vc_scan_at for
#   incremental behavior, initialize run_stats, and bind dedup helpers
#   (prefer 029 make_dedup_key/normalize_url; fallback to local hash).
#
# Inputs / Outputs:
#   Inputs:  config, (optional) get_lookback_start_date/get_lookback_end_date,
#            get_state/update_state, (optional) make_dedup_key/normalize_url
#   Outputs: scanner_config, window_start/window_end (YYYY-MM-DD),
#            freshness_cutoff_dt, last_scan_dt, run_stats, compute_dedup_key()
#
# Notes:
#   - Downstream reads config defensively (.get + defaults); no get_config().
#   - Window filter default ON via scanner_config.setdefault("enable_window_filter", True).
#   - Dedup is deterministic: prefer URL-based; fallback uses sha256.
#

from datetime import datetime, timedelta, timezone, date
from typing import Any, Dict
import hashlib

# ------------------------------------------------------------
# Helpers: safe config reads
# ------------------------------------------------------------
def _cfg_get(path: str, default=None):
    """
    Safely read nested config keys from dict using dot-path, e.g. 'limits.max_items_per_query'.
    """
    if not isinstance(config, dict):
        return default
    cur = config
    for part in path.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return default
        cur = cur[part]
    return cur

def _to_int(x, default: int) -> int:
    try:
        return int(x)
    except Exception:
        return default

# ------------------------------------------------------------
# Time window: prefer 028 helpers if available, else fallback
# ------------------------------------------------------------
get_lookback_start_date = globals().get("get_lookback_start_date")
get_lookback_end_date   = globals().get("get_lookback_end_date")

# Optional override: vc_freshness_hours (24–48h typical)
freshness_hours = _to_int(_cfg_get("monitoring.vc.freshness_hours", None), default=48)

today_utc = datetime.now(timezone.utc).date()

if callable(get_lookback_start_date) and callable(get_lookback_end_date):
    # If you later define a dedicated key like "monitoring_vc", switch here.
    # For now, reuse the daily window from 028 (currently 7 days in your config output).
    window_start = str(get_lookback_start_date("daily_notes"))
    window_end   = str(get_lookback_end_date("daily_notes"))
else:
    # Fallback: use freshness_hours (default 48h) as a rolling window mapped to dates
    window_end = today_utc.isoformat()
    window_start = (today_utc - timedelta(days=max(1, (freshness_hours + 23) // 24))).isoformat()

freshness_cutoff_dt = datetime.now(timezone.utc) - timedelta(hours=freshness_hours)

logger.info(f"[{run_id}] VC scan window: {window_start} -> {window_end}")
logger.info(f"[{run_id}] Freshness cutoff (rolling): {freshness_hours}h -> {freshness_cutoff_dt.isoformat()}")

# ------------------------------------------------------------
# Scanner defaults (config-first, safe fallbacks)
# ------------------------------------------------------------
scanner_config = {
    "category": "VC",
    "window_start": window_start,
    "window_end": window_end,
    "freshness_hours": freshness_hours,
    "timeout_sec": _to_int(_cfg_get("monitoring.vc.timeout_sec", None), 15),
    "user_agent": _cfg_get("monitoring.vc.user_agent", "Mozilla/5.0 (compatible; VCMonitor/1.0)"),
    "max_items_per_target": _to_int(_cfg_get("limits.max_items_per_query", None), 100),
    # Keep dedup strategy simple and deterministic; prefer URL-based.
    "dedup_strategy": _cfg_get("monitoring.vc.dedup_strategy", "url"),
}

logger.info(
    f"[{run_id}] Scanner defaults: timeout={scanner_config['timeout_sec']}s, "
    f"max_items_per_target={scanner_config['max_items_per_target']}, "
    f"dedup={scanner_config['dedup_strategy']}"
)

# default: window filtering ON
scanner_config.setdefault("enable_window_filter", True)

# ------------------------------------------------------------
# State: last run timestamp (incremental scanning optional)
# ------------------------------------------------------------
last_vc_scan_at = get_state("last_vc_scan_at", None)
last_scan_dt = None
if last_vc_scan_at:
    try:
        last_scan_dt = datetime.fromisoformat(str(last_vc_scan_at))
        logger.info(f"[{run_id}] Last VC scan: {last_scan_dt.isoformat()}")
    except Exception as e:
        logger.warning(f"[{run_id}] Could not parse last_vc_scan_at={last_vc_scan_at!r}: {e}; treating as first run")
        last_scan_dt = None
else:
    logger.info(f"[{run_id}] No prior VC scan found; treating as first run")

# ------------------------------------------------------------
# Run statistics (counters)
# ------------------------------------------------------------
run_stats = {
    "targets_processed": 0,
    "targets_failed": 0,
    "items_fetched": 0,
    "events_created": 0,
    "events_skipped_duplicate": 0,
    "events_skipped_stale": 0,
    "events_failed": 0,
    "start_time": datetime.now(timezone.utc).isoformat(),
    "end_time": None,
}
logger.info(f"[{run_id}] Run stats initialized: {run_stats['start_time']}")

# ------------------------------------------------------------
# Dedup key generator (prefer 029 helpers, fallback to local)
# ------------------------------------------------------------
make_dedup_key = globals().get("make_dedup_key")
normalize_url = globals().get("normalize_url")

def compute_dedup_key(event_data: Dict[str, Any]) -> str:
    """
    Deterministic dedup key for an Event.
    Prefer: make_dedup_key("event", normalize_url(url))
    Fallback: sha256(normalized_url) or sha256(title::date::source)
    """
    url = (event_data.get("url") or event_data.get("source_url") or "").strip()
    title = (event_data.get("title") or event_data.get("name") or "").strip()
    date_str = (event_data.get("date") or event_data.get("published_date") or "").strip()
    source = (event_data.get("source") or "").strip()

    # Preferred: URL-based key via 029
    if url and callable(make_dedup_key):
        u = normalize_url(url) if callable(normalize_url) else url.strip().lower()
        try:
            return make_dedup_key("event", u)
        except Exception:
            pass

    # Fallback URL hash
    if url:
        u = url.strip().lower()
        return hashlib.sha256(u.encode("utf-8")).hexdigest()

    # Fallback semantic hash
    base = f"{title}::{date_str}::{source}".strip().lower()
    return hashlib.sha256(base.encode("utf-8")).hexdigest()


2026-01-29 09:58:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] VC scan window: 2026-01-22 -> 2026-01-29
2026-01-29 09:58:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Freshness cutoff (rolling): 48h -> 2026-01-27T00:58:11.629319+00:00
2026-01-29 09:58:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Scanner defaults: timeout=15s, max_items_per_target=100, dedup=url
2026-01-29 09:58:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] No prior VC scan found; treating as first run
2026-01-29 09:58:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Run stats initialized: 2026-01-29T00:58:11.635684+00:00


In [18]:
# ============================================================
# Cell 03 — Load VC Monitoring Targets via 029
# ============================================================
# Overview:
#   Load enabled Monitoring Targets via 029.get_enabled_targets(), then filter to VC-only
#   (Type="VC", Status="ACTIVE" or empty). Normalize Source URLs into list[str] with
#   newline/comma tolerant parsing and order-preserving dedup.
#
# Inputs / Outputs:
#   Inputs:  get_enabled_targets() -> list[dict] (029 export)
#   Outputs: vc_targets: list[dict] with validated `urls` and canonical fields
#
# Notes:
#   - Never call Notion directly; use 029 wrappers only.
#   - Source URLs are rich_text in Notion; 029 flattens to a single string.
#   - Skip targets with no valid URLs; default unsupported source_type to HTML.
#

from collections import defaultdict
from typing import List
import re

# ------------------------------------------------------------
# 1) Bind 029 export (get_enabled_targets)
# ------------------------------------------------------------
get_enabled_targets = globals().get("get_enabled_targets")
if not callable(get_enabled_targets):
    logger.error(
        f"[{run_id}] get_enabled_targets() not found. "
        f"Make sure 029_notion_clients_and_io.ipynb Cell 10 executed successfully."
    )
    raw_targets = []
else:
    logger.info(f"[{run_id}] Loading enabled monitoring targets via get_enabled_targets()...")
    raw_targets = get_enabled_targets() or []
    logger.info(f"[{run_id}] Loaded {len(raw_targets)} enabled target(s)")

# ------------------------------------------------------------
# 2) Helpers: safe strings + choices + URL parsing
# ------------------------------------------------------------
SUPPORTED_SOURCE_TYPES = {"RSS", "HTML", "API"}

def _as_str(x, default="") -> str:
    if x is None:
        return default
    return x if isinstance(x, str) else str(x)

def _norm_choice(x: str) -> str:
    return _as_str(x, "").strip().upper()

def _parse_urls(text_or_list) -> List[str]:
    """
    Accepts list[str] or a single string (newline/comma separated).
    Returns unique http(s) URLs in original order.
    """
    if text_or_list is None:
        return []

    if isinstance(text_or_list, list):
        text = "\n".join([_as_str(v, "") for v in text_or_list if v is not None])
    else:
        text = _as_str(text_or_list, "")

    parts = [p.strip() for p in re.split(r"[\n,]+", text) if p.strip()]
    extracted: List[str] = []
    for p in parts:
        if p.startswith("http://") or p.startswith("https://"):
            extracted.append(p)
        else:
            extracted.extend(re.findall(r"https?://[^\s)>\"]+", p))

    out, seen = [], set()
    for u in extracted:
        u = u.strip()
        if not (u.startswith("http://") or u.startswith("https://")):
            continue
        if u in seen:
            continue
        seen.add(u)
        out.append(u)
    return out

# ------------------------------------------------------------
# 3) Filter & normalize VC targets (aligned to 029 export fields)
# ------------------------------------------------------------
vc_targets = []

for idx, t in enumerate(raw_targets):
    try:
        if not isinstance(t, dict):
            logger.warning(f"[{run_id}] Skip non-dict target idx={idx}: {type(t)}")
            continue

        target_id = t.get("page_id") or t.get("id") or f"unknown_{idx}"
        name = t.get("name") or f"Target_{idx}"

        # Type == VC
        ttype = _norm_choice(t.get("type"))
        if ttype != "VC":
            continue

        # Status == ACTIVE (or empty/None)
        status = t.get("status")
        if status is not None and _norm_choice(status) not in {"ACTIVE", ""}:
            continue

        # Source type (default HTML)
        source_type = _norm_choice(t.get("source_type") or "HTML")
        if source_type not in SUPPORTED_SOURCE_TYPES:
            logger.warning(f"[{run_id}] Unsupported Source Type '{source_type}' for '{name}'; defaulting to HTML")
            source_type = "HTML"

        # Source URLs (029 flattens rich_text -> string)
        urls = _parse_urls(t.get("source_urls"))
        if not urls:
            logger.warning(f"[{run_id}] Skip (no valid Source URLs): {name}")
            continue

        vc_targets.append({
            "id": target_id,
            "name": name,
            "type": "VC",
            "status": t.get("status") or "ACTIVE",
            "enabled": True,  # guaranteed by get_enabled_targets
            "cadence": _as_str(t.get("cadence"), ""),
            "priority": t.get("priority"),
            "search_keywords": _as_str(t.get("search_keywords"), ""),
            "source_type": source_type,
            "urls": urls,
            "url": urls[0],  # convenience
            "last_checked": t.get("last_checked"),
            "next_check": t.get("next_check"),
            "error_count": t.get("error_count"),
            "last_error": _as_str(t.get("last_error"), ""),
        })

    except Exception as e:
        logger.error(f"[{run_id}] Error normalizing target idx={idx}: {e}")
        continue

logger.info(f"[{run_id}] VC targets ready: {len(vc_targets)}")

if vc_targets:
    ct = defaultdict(int)
    for t in vc_targets:
        ct[t["source_type"]] += 1
    for k, v in sorted(ct.items()):
        logger.info(f"[{run_id}]   - {k}: {v}")

logger.info(f"[{run_id}] Cell 03 complete: VC monitoring targets loaded and validated")


2026-01-29 10:01:24 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Loading enabled monitoring targets via get_enabled_targets()...
2026-01-29 10:01:25 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | Introspected 13 properties from database 2f08e0e4d1628013b322cb01d78f0de8
2026-01-29 10:01:25 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Loaded 10 enabled target(s)
2026-01-29 10:01:25 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] VC targets ready: 9
2026-01-29 10:01:25 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7]   - HTML: 9
2026-01-29 10:01:25 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Cell 03 complete: VC monitoring targets loaded and validated


In [19]:
# ============================================================
# Cell 04 — Fetch helpers (RSS / HTML / NewsAPI)
# ============================================================
# Overview:
#   Fetch items from:
#     (A) Official sources: target["urls"] (RSS or HTML list pages)
#     (B) NewsAPI (optional; may run even if source_type=HTML)
#
# Inputs / Outputs:
#   Inputs:  vc_targets, scanner_config, (optional) NEWSAPI_KEY from 028, logger/run_id
#   Outputs: fetch_items_for_target(target) -> list[dict] items:
#            title, url, summary, published, source_type, source_url, target_id/target_name, ...
#
# Notes:
#   - No Notion writes here. Best-effort: missing optional deps degrade gracefully.
#   - RSS: feedparser if available / HTML: BeautifulSoup if available / HTTP: requests if available
#   - NewsAPI: /v2/everything with window forwarding when set
#

from __future__ import annotations

from urllib.parse import urljoin, urlparse
from datetime import datetime, timezone, timedelta

# Optional deps (safe)
try:
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
except Exception:
    requests = None
    HTTPAdapter = None
    Retry = None

try:
    import feedparser
except Exception:
    feedparser = None

try:
    from bs4 import BeautifulSoup
except Exception:
    BeautifulSoup = None


# ------------------------------------------------------------
# Requests session (with retries)
# ------------------------------------------------------------
_session = None

def get_http_session():
    global _session
    if _session is not None:
        return _session
    if requests is None:
        return None

    s = requests.Session()
    retry_total = int(scanner_config.get("http_retry_total", 2))
    if Retry and HTTPAdapter and retry_total > 0:
        retry = Retry(
            total=retry_total,
            backoff_factor=float(scanner_config.get("http_retry_backoff", 0.5)),
            status_forcelist=tuple(scanner_config.get("http_retry_statuses", [429, 500, 502, 503, 504])),
            allowed_methods=frozenset(["GET"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry)
        s.mount("http://", adapter)
        s.mount("https://", adapter)

    _session = s
    return _session


# ------------------------------------------------------------
# Headers + HTTP fetch (text only)
# ------------------------------------------------------------
def build_headers(target: dict) -> dict:
    headers = {"User-Agent": scanner_config.get("user_agent", "Mozilla/5.0 (compatible; VCMonitor/1.0)")}
    if isinstance(target.get("custom_headers"), dict):
        headers.update(target["custom_headers"])
    return headers

def fetch_url_text(url: str, *, headers: dict, timeout_sec: int) -> str | None:
    if requests is None:
        logger.warning(f"[{run_id}] requests not available; cannot fetch: {url}")
        return None

    s = get_http_session()
    try:
        r = s.get(url, headers=headers, timeout=timeout_sec, allow_redirects=True)
        r.raise_for_status()

        ctype = (r.headers.get("Content-Type") or "").lower()
        if ctype and not any(t in ctype for t in ["text/", "html", "xml", "json", "rss", "atom"]):
            logger.debug(f"[{run_id}] Skipping non-text content-type={ctype}: {url}")
            return None

        if not r.encoding:
            r.encoding = r.apparent_encoding
        return r.text

    except Exception as e:
        logger.warning(f"[{run_id}] fetch failed: {url} ({type(e).__name__}: {e})")
        return None


# ------------------------------------------------------------
# RSS parsing (feedparser)
# ------------------------------------------------------------
def parse_rss_entries(rss_text: str, *, target: dict, source_url: str) -> list[dict]:
    if feedparser is None:
        logger.warning(f"[{run_id}] feedparser not available; cannot parse RSS for {target.get('name')}")
        return []

    feed = feedparser.parse(rss_text)
    entries = feed.get("entries", []) or []
    if not entries:
        return []

    items = []
    max_items = int(scanner_config.get("max_items_per_target", 100))

    for e in entries[:max_items]:
        try:
            title = (e.get("title") or "").strip() or "Untitled"
            link = (e.get("link") or "").strip()
            if not link:
                continue

            summary = (e.get("summary") or e.get("description") or "").strip()
            published = (e.get("published") or e.get("updated") or "").strip()

            items.append({
                "title": title,
                "url": link,
                "summary": summary,
                "published": published,
                "source_name": target.get("name"),
                "source_url": source_url,
                "source_type": "RSS",
                "target_id": target.get("id"),
                "target_name": target.get("name"),
                "search_keywords": target.get("search_keywords", ""),
            })
        except Exception as ex:
            logger.debug(f"[{run_id}] RSS entry parse error ({target.get('name')}): {ex}")
            continue

    return items


# ------------------------------------------------------------
# HTML link extraction (heuristic)
# ------------------------------------------------------------
def extract_html_links(html: str, *, target: dict, source_url: str) -> list[dict]:
    if BeautifulSoup is None:
        logger.warning(f"[{run_id}] BeautifulSoup not available; cannot parse HTML for {target.get('name')}")
        return []

    soup = BeautifulSoup(html, "html.parser")
    scope = soup.find("main") or soup.find("article") or soup.find("body") or soup

    for tag in scope.find_all(["nav", "footer", "header", "aside"]):
        try:
            tag.decompose()
        except Exception:
            pass

    anchors = scope.find_all("a", href=True)
    candidates, seen = [], set()

    max_items = int(scanner_config.get("max_items_per_target", 100))
    base_netloc = urlparse(source_url).netloc.lower()

    def _looks_like_article_url(u: str) -> bool:
        low = u.lower()
        if any(x in low for x in ["/tag/", "/tags/", "/category/", "/categories/", "/about", "/contact", "/privacy", "/terms"]):
            return False
        return True

    for a in anchors:
        if len(candidates) >= max_items:
            break

        text = (a.get_text(" ", strip=True) or "").strip()
        href = (a.get("href") or "").strip()
        if not href:
            continue

        if href.startswith("#") or href.startswith("mailto:") or href.startswith("javascript:"):
            continue
        if not text or len(text) < 8:
            continue

        abs_url = href if href.startswith("http") else urljoin(source_url, href)
        u = urlparse(abs_url)

        # same-site preference (noise reduction)
        if u.netloc and base_netloc and u.netloc.lower() != base_netloc:
            continue
        if not _looks_like_article_url(abs_url):
            continue

        key = abs_url.lower()
        if key in seen:
            continue
        seen.add(key)

        summary = ""
        try:
            parent = a.find_parent(["article", "li", "div", "section"])
            if parent:
                p = parent.find("p")
                if p:
                    summary = (p.get_text(" ", strip=True) or "").strip()
            summary = summary[:400]
        except Exception:
            summary = ""

        candidates.append({
            "title": text,
            "url": abs_url,
            "summary": summary,
            "published": "",
            "source_name": target.get("name"),
            "source_url": source_url,
            "source_type": "HTML",
            "target_id": target.get("id"),
            "target_name": target.get("name"),
            "search_keywords": target.get("search_keywords", ""),
        })

    return candidates


# ------------------------------------------------------------
# NewsAPI (/v2/everything)
# ------------------------------------------------------------
def fetch_items_from_newsapi(target: dict) -> list[dict]:
    """
    Runs NewsAPI even if target["source_type"] == "HTML".
    Query: target["newsapi"]["q"] -> target["search_keywords"]
    Window: scanner_config["window_start"/"window_end"] else rolling days (newsapi_days)
    """
    if requests is None:
        logger.warning(f"[{run_id}] requests not available; cannot call NewsAPI for {target.get('name')}")
        return []

    if not bool(scanner_config.get("enable_newsapi", True)):
        return []
    if bool(target.get("disable_newsapi", False)):
        return []

    api_key = (
        (scanner_config.get("newsapi_api_key") or "").strip()
        or (scanner_config.get("newsapi_key") or "").strip()
        or (globals().get("NEWSAPI_KEY") or "").strip()  # from 028
    )
    if not api_key:
        logger.info(f"[{run_id}] NewsAPI key missing; skip NewsAPI for {target.get('name')}")
        return []

    news_cfg = target.get("newsapi") or {}
    q = (news_cfg.get("q") or target.get("search_keywords") or "").strip()
    if not q:
        logger.info(f"[{run_id}] NewsAPI skipped (no query/search_keywords): {target.get('name')}")
        return []

    window_end = (scanner_config.get("window_end") or "").strip()
    window_start = (scanner_config.get("window_start") or "").strip()

    if not window_end:
        window_end = datetime.now(timezone.utc).date().isoformat()

    if not window_start:
        days = int(scanner_config.get("newsapi_days", 14))
        dt_end = datetime.fromisoformat(window_end).replace(tzinfo=timezone.utc)
        window_start = (dt_end - timedelta(days=days)).date().isoformat()

    endpoint = (scanner_config.get("newsapi_everything_url") or "https://newsapi.org/v2/everything").strip()
    timeout_sec = int(scanner_config.get("timeout_sec", 15))

    page_size = int(news_cfg.get("pageSize") or scanner_config.get("newsapi_page_size", 50))
    page_size = max(1, min(100, page_size))
    max_pages = int(news_cfg.get("max_pages") or scanner_config.get("newsapi_max_pages", 1))
    max_pages = max(1, min(10, max_pages))

    params_base = {
        "q": q,
        "from": window_start,
        "to": window_end,
        "language": news_cfg.get("language") or scanner_config.get("newsapi_language", "en"),
        "sortBy": news_cfg.get("sortBy") or scanner_config.get("newsapi_sort_by", "publishedAt"),
        "domains": news_cfg.get("domains"),
        "excludeDomains": news_cfg.get("excludeDomains"),
        "sources": news_cfg.get("sources"),
        "searchIn": news_cfg.get("searchIn"),
        "pageSize": page_size,
        "apiKey": api_key,
    }
    params_base = {k: v for k, v in params_base.items() if v not in [None, "", []]}

    headers = build_headers(target)
    s = get_http_session()

    logger.info(
        f"[{run_id}] [NEWSAPI_CALL] target={target.get('name')} "
        f"q='{q[:80]}' from={window_start} to={window_end} pages={max_pages} pageSize={page_size}"
    )

    items: list[dict] = []
    for page in range(1, max_pages + 1):
        try:
            params = dict(params_base)
            params["page"] = page

            r = s.get(endpoint, params=params, headers=headers, timeout=timeout_sec)
            if r.status_code != 200:
                logger.warning(f"[{run_id}] [NEWSAPI] non-200 status={r.status_code} body(head)={r.text[:300]}")
                r.raise_for_status()

            data = r.json() if "json" in (r.headers.get("Content-Type", "").lower()) else {}
            if not isinstance(data, dict) or data.get("status") != "ok":
                logger.warning(f"[{run_id}] [NEWSAPI] bad response keys={list(data.keys())[:10] if isinstance(data, dict) else 'n/a'}")
                break

            arts = data.get("articles") or []
            if not arts:
                break

            for a in arts:
                try:
                    title = (a.get("title") or "").strip()
                    url = (a.get("url") or "").strip()
                    if not title or not url:
                        continue

                    items.append({
                        "title": title,
                        "url": url,
                        "summary": (a.get("description") or a.get("content") or "").strip(),
                        "published": (a.get("publishedAt") or "").strip(),  # used in Cell 05
                        "source_name": target.get("name"),
                        "source_url": f"newsapi:everything?q={q}",
                        "source_type": "NEWSAPI",
                        "target_id": target.get("id"),
                        "target_name": target.get("name"),
                        "search_keywords": target.get("search_keywords", ""),
                    })
                except Exception:
                    continue

            if len(arts) < page_size:
                break

        except Exception as e:
            logger.warning(f"[{run_id}] [NEWSAPI_ERR] target={target.get('name')} ({type(e).__name__}: {e})")
            break

    logger.info(f"[{run_id}] [NEWSAPI_DONE] target={target.get('name')} items={len(items)}")
    return items


# ------------------------------------------------------------
# Unified per-source-url fetch
# ------------------------------------------------------------
def fetch_items_for_source_url(target: dict, source_url: str) -> list[dict]:
    st = (target.get("source_type") or "HTML").upper().strip()
    logger.info(f"[{run_id}] [FETCH] target={target.get('name')} st={st} url={source_url}")

    timeout_sec = int(scanner_config.get("timeout_sec", 15))
    headers = build_headers(target)

    text = fetch_url_text(source_url, headers=headers, timeout_sec=timeout_sec)
    if not text:
        return []

    if st == "RSS":
        return parse_rss_entries(text, target=target, source_url=source_url)

    # NOTE: even if HTML, NewsAPI is handled in fetch_items_for_target()
    return extract_html_links(text, target=target, source_url=source_url)


# ------------------------------------------------------------
# Target-level fetch (multiple urls + NewsAPI + light URL-dedup)
# ------------------------------------------------------------
def fetch_items_for_target(target: dict) -> list[dict]:
    """
    (A) official urls (RSS/HTML)
    (B) NewsAPI (if enabled + key exists)
    (C) dedup by URL (prefer item with published)
    """
    all_items: list[dict] = []

    for u in (target.get("urls") or []):
        all_items.extend(fetch_items_for_source_url(target, u))

    all_items.extend(fetch_items_from_newsapi(target))

    dedup: dict[str, dict] = {}
    for it in all_items:
        key = (it.get("url") or "").strip().lower()
        if not key:
            continue
        if key not in dedup:
            dedup[key] = it
        else:
            cur_pub = (dedup[key].get("published") or "").strip()
            new_pub = (it.get("published") or "").strip()
            if (not cur_pub) and new_pub:
                dedup[key] = it

    return list(dedup.values())


logger.info(f"[{run_id}] Cell 04 complete: fetch helpers ready (RSS/HTML/NEWSAPI, url-list aware)")


2026-01-29 10:06:56 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Cell 04 complete: fetch helpers ready (RSS/HTML/NEWSAPI, url-list aware)


In [20]:
# ============================================================
# Cell 05 — Normalize items into Event candidates (enrich-ready, debug-safe) — REPLACEMENT
# ============================================================
# Fixes / Guarantees:
#   - NameError safety: ensure all item references live inside the raw_items loop.
#   - Safe per-item aggregation for source_type counts + NEWSAPI sample logs.
#   - Stats updated without breaking the pipeline (processed/failed/items_fetched/etc.).
#   - Date detection + freshness filter + optional window filter preserved.
#
# Assumes:
#   - fetch_items_for_target(target) exists (Cell 04)
#   - compute_dedup_key, compute_next_check, mark_target_checked exist
#   - vc_targets, scanner_config, logger, run_id exist
#

from __future__ import annotations

import re
from datetime import datetime, timezone, date, timedelta
from typing import Optional, Tuple

today = datetime.now(timezone.utc).date()
today_iso = today.isoformat()

# NOTE: Keep behavior as provided (local run_stats is used here)
# scanner_config["enable_window_filter"] = False
# scanner_config["freshness_days"] = 3650
run_stats = {}

# ------------------------------------------------------------
# Freshness cutoff (days-based)
# ------------------------------------------------------------
freshness_days = int(scanner_config.get("freshness_days", 14))
freshness_cutoff_dt = datetime.now(timezone.utc) - timedelta(days=freshness_days)

logger.info(
    f"[{run_id}] freshness_cutoff_dt={freshness_cutoff_dt.isoformat()} "
    f"freshness_days={freshness_days}"
)
logger.info(
    f"[{run_id}] freshness_cutoff_dt={freshness_cutoff_dt.isoformat()} "
    f"today_utc={datetime.now(timezone.utc).isoformat()} "
    f"freshness_days={scanner_config.get('freshness_days')}"
)

# ------------------------------------------------------------
# Debug flags (bounded)
# ------------------------------------------------------------
DEBUG_DATE = bool(scanner_config.get("debug_date", True))
DEBUG_DATE_MAX = int(scanner_config.get("debug_date_max", 30))
_debug_date_count = 0

DEBUG_NEWSAPI = bool(scanner_config.get("debug_newsapi_seen", True))
DEBUG_NEWSAPI_MAX = int(scanner_config.get("debug_newsapi_seen_max", 10))

# Optional date parsing (RSS/API published strings)
try:
    from dateutil import parser as dateutil_parser  # type: ignore
except Exception:
    dateutil_parser = None

event_candidates: list[dict] = []

# Ensure run_stats has expected debug buckets
run_stats.setdefault("_source_type_counts", {})
run_stats.setdefault("_newsapi_seen", 0)

logger.info(f"[{run_id}] Starting normalization for {len(vc_targets)} VC target(s)...")

# ------------------------------------------------------------
# Window bounds (optional)
# ------------------------------------------------------------
def _parse_date_yyyy_mm_dd(s) -> Optional[date]:
    try:
        return date.fromisoformat(str(s))
    except Exception:
        return None

window_start_date = _parse_date_yyyy_mm_dd(scanner_config.get("window_start"))
window_end_date = _parse_date_yyyy_mm_dd(scanner_config.get("window_end"))

# ------------------------------------------------------------
# Parse published datetime (RSS/API strings)
# ------------------------------------------------------------
def _parse_published_dt(published_str: str) -> Optional[datetime]:
    if not published_str:
        return None
    s = str(published_str).strip()
    if not s:
        return None

    # ISO first
    try:
        dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt
    except Exception:
        pass

    # dateutil fallback
    if dateutil_parser:
        try:
            dt = dateutil_parser.parse(s)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt
        except Exception:
            return None

    return None

# ------------------------------------------------------------
# Extract date from title/summary text (regex)
# ------------------------------------------------------------
_MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

def _safe_dt(y: int, m: int, d: int) -> Optional[datetime]:
    try:
        return datetime(y, m, d, tzinfo=timezone.utc)
    except Exception:
        return None

def _extract_date_from_text(text: str) -> Tuple[Optional[datetime], str]:
    """
    Returns (dt, reason) describing match path.
    Supports:
      1) YYYY-MM-DD / YYYY/MM/DD / YYYY.MM.DD
      2) YYYY年M月D日
      3) Month D, YYYY (January 5, 2026)
      4) D Month YYYY (5 January 2026)
    """
    if not text:
        return None, "empty"
    t = " ".join(str(text).split())
    if not t:
        return None, "blank"

    # 1) YYYY-MM-DD / YYYY/MM/DD / YYYY.MM.DD
    m = re.search(r"\b(20\d{2})[-/.](\d{1,2})[-/.](\d{1,2})\b", t)
    if m:
        dt = _safe_dt(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        return dt, f"ymd_sep:{m.group(0)}"

    # 2) YYYY年M月D日
    m = re.search(r"\b(20\d{2})\s*年\s*(\d{1,2})\s*月\s*(\d{1,2})\s*日\b", t)
    if m:
        dt = _safe_dt(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        return dt, f"jp_ymd:{m.group(0)}"

    # 3) Month D, YYYY
    m = re.search(r"\b([A-Za-z]{3,9})\s+(\d{1,2})(?:st|nd|rd|th)?[,]?\s+(20\d{2})\b", t)
    if m:
        mon = _MONTHS.get(m.group(1).lower())
        if mon:
            dt = _safe_dt(int(m.group(3)), mon, int(m.group(2)))
            return dt, f"mon_d_yyyy:{m.group(0)}"
        return None, f"mon_unknown:{m.group(1)}"

    # 4) D Month YYYY
    m = re.search(r"\b(\d{1,2})\s+([A-Za-z]{3,9})[,]?\s+(20\d{2})\b", t)
    if m:
        mon = _MONTHS.get(m.group(2).lower())
        if mon:
            dt = _safe_dt(int(m.group(3)), mon, int(m.group(1)))
            return dt, f"d_mon_yyyy:{m.group(0)}"
        return None, f"mon_unknown:{m.group(2)}"

    return None, "no_match"

# ------------------------------------------------------------
# Main loop: targets -> items -> normalized candidates
# ------------------------------------------------------------
for target in vc_targets:
    target_name = target.get("name", "unknown")
    target_id = target.get("id")  # Notion page_id of Monitoring Target

    try:
        logger.info(f"[{run_id}] Fetching items for target: {target_name}")
        raw_items = fetch_items_for_target(target)

        # processed = attempted fetch
        run_stats["targets_processed"] = run_stats.get("targets_processed", 0) + 1

        if not raw_items:
            logger.info(f"[{run_id}] No items fetched: {target_name}")
        else:
            run_stats["items_fetched"] = run_stats.get("items_fetched", 0) + len(raw_items)
            logger.info(f"[{run_id}] Fetched {len(raw_items)} item(s): {target_name}")

            for item in raw_items:
                try:
                    # ---- Source-type counts (per item) ----
                    st = (item.get("source_type") or target.get("source_type") or "").upper().strip() or "UNKNOWN"
                    run_stats["_source_type_counts"][st] = run_stats["_source_type_counts"].get(st, 0) + 1

                    # Periodic sample log (optional)
                    if sum(run_stats["_source_type_counts"].values()) % 200 == 0:
                        logger.info(f"[{run_id}] [SRC_TYPE_COUNTS] {run_stats['_source_type_counts']}")

                    # NewsAPI sample (bounded)
                    if DEBUG_NEWSAPI and st == "NEWSAPI" and run_stats.get("_newsapi_seen", 0) < DEBUG_NEWSAPI_MAX:
                        run_stats["_newsapi_seen"] = run_stats.get("_newsapi_seen", 0) + 1
                        logger.info(
                            f"[{run_id}] [NEWSAPI_SEEN] "
                            f"published='{(item.get('published') or '')[:40]}' "
                            f"title='{(item.get('title') or '')[:80]}' "
                            f"url='{item.get('url')}'"
                        )

                    # ---- Normalize fields ----
                    title = (item.get("title") or "").strip()
                    url = (item.get("url") or "").strip()
                    summary = (item.get("summary") or "").strip()
                    published_str = item.get("published") or ""

                    if not title or not url:
                        continue

                    # ---- Determine published_dt ----
                    why_t = "n/a"
                    why_s = "n/a"

                    published_dt = _parse_published_dt(str(published_str))
                    published_known = published_dt is not None
                    date_source = "published" if published_known else ""

                    if not published_known:
                        dt_t, why_t = _extract_date_from_text(title)
                        dt_s, why_s = _extract_date_from_text(summary)

                        guess_dt = dt_t or dt_s
                        if guess_dt is not None:
                            published_dt = guess_dt
                            published_known = True
                            date_source = "title_or_summary"
                        else:
                            # Drop items where date cannot be detected
                            run_stats["events_skipped_no_date"] = run_stats.get("events_skipped_no_date", 0) + 1
                            if DEBUG_DATE and _debug_date_count < DEBUG_DATE_MAX:
                                _debug_date_count += 1
                                logger.info(
                                    f"[{run_id}] [DATE_DROP] no date detected | "
                                    f"title='{title[:80]}' | url={url}"
                                )
                            continue

                    # Debug log (bounded)
                    if DEBUG_DATE and _debug_date_count < DEBUG_DATE_MAX:
                        _debug_date_count += 1
                        logger.info(
                            f"[{run_id}] [DATE_DEBUG] title='{title[:80]}' | "
                            f"published_str='{str(published_str)[:80]}' | "
                            f"title_match={why_t} | summary_match={why_s} | "
                            f"chosen_source={date_source} | chosen_date={published_dt.date().isoformat()} | "
                            f"url={url}"
                        )

                    # ---- Freshness filtering ----
                    if published_dt and published_dt < freshness_cutoff_dt:
                        run_stats["events_skipped_stale"] = run_stats.get("events_skipped_stale", 0) + 1
                        continue

                    # ---- Window filtering (optional) ----
                    enable_window = bool(scanner_config.get("enable_window_filter", True))
                    if enable_window and published_dt and window_start_date and window_end_date:
                        d = published_dt.date()
                        if d < window_start_date or d > window_end_date:
                            run_stats["events_skipped_stale"] = run_stats.get("events_skipped_stale", 0) + 1
                            continue

                    if not summary:
                        summary = ""

                    candidate = {
                        "title": title,
                        "url": url,

                        # Dates
                        "published_dt": published_dt,
                        "date": published_dt.date().isoformat(),  # Notion date
                        "published_known": True,
                        "date_source": date_source,

                        # Content
                        "summary": summary,

                        # Provenance
                        "source": target_name,
                        "source_id": target_id,
                        "source_type": item.get("source_type") or target.get("source_type"),
                        "source_url": item.get("source_url") or "",

                        # Target info
                        "target_name": target_name,
                        "target_page_id": target_id,
                        "priority": target.get("priority"),
                        "search_keywords": target.get("search_keywords", ""),

                        # Minimal raw debug context
                        "raw_item_min": {
                            "published": item.get("published", ""),
                            "source_type": item.get("source_type", ""),
                            "source_url": item.get("source_url", ""),
                        },
                    }

                    candidate["dedup_key"] = compute_dedup_key({
                        "url": candidate["url"],
                        "title": candidate["title"],
                        "date": candidate["date"],
                        "source": candidate["source"],
                    })

                    event_candidates.append(candidate)

                except Exception as e:
                    run_stats["events_failed"] = run_stats.get("events_failed", 0) + 1
                    logger.debug(f"[{run_id}] Normalize item error ({target_name}): {type(e).__name__}: {e}")
                    continue

        # Target-level scheduling update (cadence -> next_check), isolated from main loop
        try:
            cadence = target.get("cadence")
            next_check = compute_next_check(cadence, today)  # today: date
            if callable(mark_target_checked) and target_id:
                mark_target_checked(
                    target_id,
                    checked_date=today_iso,
                    next_check=next_check.isoformat() if next_check else None,
                )
        except Exception as e:
            logger.warning(
                f"[{run_id}] Target scheduling update failed: {target_name} "
                f"({type(e).__name__}: {e})"
            )

    except Exception as e:
        run_stats["targets_failed"] = run_stats.get("targets_failed", 0) + 1
        logger.warning(f"[{run_id}] Target processing failed: {target_name} ({type(e).__name__}: {e})")
        continue

# Sort newest first
if event_candidates:
    event_candidates.sort(key=lambda x: x["published_dt"], reverse=True)

logger.info(f"[{run_id}] Normalization complete: {len(event_candidates)} candidate(s)")
logger.info(
    f"[{run_id}] Stats: "
    f"targets_processed={run_stats.get('targets_processed',0)}, "
    f"targets_failed={run_stats.get('targets_failed',0)}, "
    f"items_fetched={run_stats.get('items_fetched',0)}, "
    f"events_skipped_no_date={run_stats.get('events_skipped_no_date',0)}, "
    f"events_skipped_stale={run_stats.get('events_skipped_stale',0)}, "
    f"events_failed={run_stats.get('events_failed',0)}"
)
logger.info(f"[{run_id}] [SRC_TYPE_COUNTS_FINAL] {run_stats.get('_source_type_counts')}")
logger.info(f"[{run_id}] Cell 05 complete: candidates ready for enrichment (Cell 05.5)")


2026-01-29 10:10:06 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] freshness_cutoff_dt=2026-01-15T01:10:06.399943+00:00 freshness_days=14
2026-01-29 10:10:06 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] freshness_cutoff_dt=2026-01-15T01:10:06.399943+00:00 today_utc=2026-01-29T01:10:06.485016+00:00 freshness_days=None
2026-01-29 10:10:06 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Starting normalization for 9 VC target(s)...
2026-01-29 10:10:06 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Fetching items for target: B Capital
2026-01-29 10:10:06 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] [FETCH] target=B Capital st=HTML url=https://b.capital/news-and-insights/
2026-01-29 10:10:07 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-25847

In [21]:
# ============================================================
# Cell 05.25 — Pre-enrichment Notion dedup (skip already-existing candidates)
# ============================================================
# Overview:
#   Before enrichment (Cell 05.5), remove candidates that already exist in Notion
#   by dedup_key. This reduces cost/time by avoiding enrichment work on duplicates.
#
# Strategy:
#   1) Fast path: load dedup_keys from recent Events via 029.get_recent_events()
#   2) Fallback: per-key lookup via 029.query_event_by_dedup_key()
#      (only if candidate volume is small and fast path produced no keys)
#
# Inputs / Outputs:
#   Inputs:  event_candidates, scanner_config, run_stats, (029) get_recent_events/query_event_by_dedup_key
#   Outputs: event_candidates (reassigned, filtered), run_stats["events_skipped_duplicate"] incremented
#
# Notes:
#   - Best-effort: if Notion queries fail, keep all candidates.
#   - Scanning "recent" events is usually sufficient for duplicates in monitoring flows.
#

# 029 wrappers (executed via %run -i)
get_recent_events = globals().get("get_recent_events")
query_event_by_dedup_key = globals().get("query_event_by_dedup_key")

# Config
enable_pre_dedup = bool(scanner_config.get("enable_pre_dedup", True))
max_recent_events_scan = int(scanner_config.get("max_recent_events_scan", 500))
per_key_query_threshold = int(scanner_config.get("per_key_query_threshold", 100))

if not enable_pre_dedup:
    logger.info(f"[{run_id}] Pre-dedup disabled; will enrich all candidates")
else:
    existing_keys: set[str] = set()

    # ------------------------------------------------------------
    # Fast path: cache dedup_keys from recent Events
    # ------------------------------------------------------------
    if callable(get_recent_events):
        try:
            recent = get_recent_events(limit=max_recent_events_scan)
            for ev in (recent or []):
                dk = (ev.get("dedup_key") or "").strip()
                if dk:
                    existing_keys.add(dk)
            logger.info(f"[{run_id}] Loaded dedup_keys from recent events: {len(existing_keys)}")
        except Exception as e:
            logger.warning(f"[{run_id}] get_recent_events failed: {type(e).__name__}: {e}")

    # ------------------------------------------------------------
    # Fallback: per-key queries (only when candidate volume is small)
    # ------------------------------------------------------------
    if (not existing_keys) and callable(query_event_by_dedup_key) and len(event_candidates) <= per_key_query_threshold:
        try:
            hits = 0
            for c in event_candidates:
                dk = (c.get("dedup_key") or "").strip()
                if not dk:
                    continue
                ex = query_event_by_dedup_key(dk)
                if ex:
                    existing_keys.add(dk)
                    hits += 1
            logger.info(f"[{run_id}] Built existing_keys via per-key query: {len(existing_keys)} (hits={hits})")
        except Exception as e:
            logger.warning(f"[{run_id}] Per-key dedup query failed: {type(e).__name__}: {e}")

    # ------------------------------------------------------------
    # Apply filtering
    # ------------------------------------------------------------
    before = len(event_candidates)
    if existing_keys:
        event_candidates = [
            c for c in event_candidates
            if (c.get("dedup_key") or "").strip() not in existing_keys
        ]
        skipped = before - len(event_candidates)
        run_stats["events_skipped_duplicate"] = run_stats.get("events_skipped_duplicate", 0) + skipped
        logger.info(f"[{run_id}] Pre-dedup removed {skipped} candidate(s); remaining={len(event_candidates)}")
    else:
        logger.info(f"[{run_id}] No existing_keys available; will enrich all candidates")


2026-01-29 10:14:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | Introspected 15 properties from database 2f08e0e4d16280beb40cf607bbb3b828
2026-01-29 10:14:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Loaded dedup_keys from recent events: 40
2026-01-29 10:14:11 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Pre-dedup removed 3 candidate(s); remaining=0


In [22]:
# ============================================================
# Cell 05.5 — Enrich articles (robots → fetch HTML → extract text → OpenAI ~100-word EN summary)
# ============================================================
# Overview:
#   Enrich event candidates by (1) checking robots.txt, (2) fetching article HTML,
#   (3) extracting a compact text snippet, and (4) generating a ~100-word English summary via OpenAI.
#   Updates candidate["summary"] and candidate["confidence"] before Cell 06 writes to Notion.
#
# Inputs / Outputs:
#   Inputs:  event_candidates, scanner_config, fetch_url_text() (Cell 04)
#   Outputs: In-place updates on candidates:
#     - summary (filled/overwritten when enrichment succeeds)
#     - confidence (float)
#     - enrich_status, robots_allowed, robots_reason, extracted_text_len, summary_source, llm_status
#
# Notes:
#   - Best-effort robots compliance:
#       - If robots cannot be fetched/parsed -> allow (fail-open) but record the reason
#       - If robots disallows -> skip enrichment for that URL
#   - Cost control via max_enrich_items
#   - No Notion writes here
#

from __future__ import annotations

from urllib.parse import urlparse
from urllib import robotparser
from collections import defaultdict

# Optional dependency
try:
    from bs4 import BeautifulSoup
except Exception:
    BeautifulSoup = None

# ----------------------------
# Config (defensive)
# ----------------------------
max_enrich_items = int(scanner_config.get("max_enrich_items", 1000))  # cost control
article_timeout_sec = int(scanner_config.get("article_timeout_sec", scanner_config.get("timeout_sec", 15)))
max_extract_chars = int(scanner_config.get("max_extract_chars", 3000))
min_extract_chars_for_llm = int(scanner_config.get("min_extract_chars_for_llm", 400))

enrich_only_if_missing_summary = bool(scanner_config.get("enrich_only_if_missing_summary", True))
write_enriched_into_summary = bool(scanner_config.get("write_enriched_into_summary", True))

UA = scanner_config.get("user_agent", "Mozilla/5.0 (compatible; VCMonitor/1.0)")

# ----------------------------
# Robots cache (per robots.txt URL)
# ----------------------------
_robots_cache: dict[str, robotparser.RobotFileParser | None] = {}

def _robots_url_for(u: str) -> str | None:
    try:
        p = urlparse(u)
        if not p.scheme or not p.netloc:
            return None
        return f"{p.scheme}://{p.netloc}/robots.txt"
    except Exception:
        return None

def is_allowed_by_robots(article_url: str, user_agent: str) -> tuple[bool, str]:
    """
    Returns (allowed, reason).

    Policy:
      - If robots fetch/parse fails -> allow (fail-open) with reason
      - Otherwise follow rp.can_fetch
    """
    rurl = _robots_url_for(article_url)
    if not rurl:
        return True, "robots_url_missing_allow"

    if rurl not in _robots_cache:
        rp = robotparser.RobotFileParser()
        rp.set_url(rurl)
        try:
            headers = {"User-Agent": user_agent}
            txt = fetch_url_text(rurl, headers=headers, timeout_sec=article_timeout_sec)
            if txt is None:
                _robots_cache[rurl] = None
            else:
                rp.parse(txt.splitlines())
                _robots_cache[rurl] = rp
        except Exception:
            _robots_cache[rurl] = None

    rp = _robots_cache.get(rurl)
    if rp is None:
        return True, "robots_unavailable_allow"

    try:
        allowed = bool(rp.can_fetch(user_agent, article_url))
        return allowed, ("robots_ok" if allowed else "robots_disallow")
    except Exception:
        return True, "robots_parse_error_allow"

# ----------------------------
# HTML text extraction (compact snippet)
# ----------------------------
def extract_article_text(html: str) -> dict:
    """
    Returns:
      {
        "meta_description": str,
        "og_description": str,
        "title": str,
        "text": str,      # compact snippet for LLM
        "text_len": int,
      }
    """
    out = {"meta_description": "", "og_description": "", "title": "", "text": "", "text_len": 0}
    if not html or BeautifulSoup is None:
        return out

    soup = BeautifulSoup(html, "html.parser")

    # Title
    try:
        if soup.title and soup.title.get_text(strip=True):
            out["title"] = soup.title.get_text(" ", strip=True)
    except Exception:
        pass

    # Meta descriptions
    try:
        og = soup.find("meta", attrs={"property": "og:description"})
        if og and og.get("content"):
            out["og_description"] = str(og.get("content")).strip()
    except Exception:
        pass

    try:
        md = soup.find("meta", attrs={"name": "description"})
        if md and md.get("content"):
            out["meta_description"] = str(md.get("content")).strip()
    except Exception:
        pass

    # Main text: article → main → body (best-effort)
    scope = soup.find("article") or soup.find("main") or soup.find("body") or soup

    # Remove noisy blocks
    for tag in scope.find_all(["script", "style", "noscript", "nav", "footer", "header", "aside"]):
        try:
            tag.decompose()
        except Exception:
            pass

    # Prefer paragraphs
    paras = []
    total = 0
    try:
        for p in scope.find_all("p"):
            t = (p.get_text(" ", strip=True) or "").strip()
            if len(t) >= 40:
                paras.append(t)
                total += len(t)
            if total >= max_extract_chars:
                break
    except Exception:
        paras = []

    text = "\n".join(paras).strip()
    if len(text) > max_extract_chars:
        text = text[:max_extract_chars]

    out["text"] = text
    out["text_len"] = len(text)
    return out

# ----------------------------
# OpenAI summarization (defensive)
# ----------------------------
def summarize_100w_english(text: str) -> tuple[str | None, str]:
    """
    Returns (summary, status)
      status in {"ok", "no_text", "openai_unavailable", "error"}
    """
    if not text or len(text.strip()) < min_extract_chars_for_llm:
        return None, "no_text"

    model = scanner_config.get("openai_model", "gpt-4o-mini")
    temperature = float(scanner_config.get("openai_temperature", 0.2))

    prompt = (
        "Summarize the following article in about 100 words in English.\n"
        "Focus on: what happened, who is involved, and why it matters for startups or venture capital.\n\n"
        f"TEXT:\n{text}"
    )

    # OpenAI SDK v1
    try:
        from openai import OpenAI  # type: ignore
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You are a neutral, factual summarizer. Do not speculate."},
                {"role": "user", "content": prompt},
            ],
        )
        summary = (resp.choices[0].message.content or "").strip()
        return (summary or None), ("ok" if summary else "error")
    except Exception:
        pass

    # Legacy openai fallback
    try:
        import openai  # type: ignore
        resp = openai.ChatCompletion.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You are a neutral, factual summarizer. Do not speculate."},
                {"role": "user", "content": prompt},
            ],
        )
        summary = (resp["choices"][0]["message"]["content"] or "").strip()
        return (summary or None), ("ok" if summary else "error")
    except Exception:
        return None, "openai_unavailable"

# ----------------------------
# Main enrichment loop (accounting + bounded cost)
# ----------------------------
logger.info(f"[{run_id}] Starting enrichment (max_enrich_items={max_enrich_items})")

enriched = 0
skipped = 0
attempted = 0
skip_reasons = defaultdict(int)

for c in event_candidates:
    if enriched >= max_enrich_items:
        break

    attempted += 1

    if enrich_only_if_missing_summary and (c.get("summary") or "").strip():
        c["enrich_status"] = "skip_has_summary"
        skipped += 1
        skip_reasons["has_summary"] += 1
        continue

    url = (c.get("url") or "").strip()
    if not url.startswith("http"):
        c["enrich_status"] = "skip_invalid_url"
        skipped += 1
        skip_reasons["invalid_url"] += 1
        continue

    # Robots check
    allowed, robots_reason = is_allowed_by_robots(url, UA)
    c["robots_allowed"] = bool(allowed)
    c["robots_reason"] = robots_reason
    if not allowed:
        c["enrich_status"] = "robots_blocked"
        c["confidence"] = float(c.get("confidence") or 0.55)
        skipped += 1
        skip_reasons["robots_blocked"] += 1
        continue

    # Fetch HTML
    try:
        headers = {"User-Agent": UA}
        html = fetch_url_text(url, headers=headers, timeout_sec=article_timeout_sec)
    except Exception as e:
        logger.debug(f"[{run_id}] Article fetch error: {type(e).__name__}: {e}")
        html = None

    if not html:
        c["enrich_status"] = "fetch_failed"
        c["confidence"] = float(c.get("confidence") or 0.50)
        skipped += 1
        skip_reasons["fetch_failed"] += 1
        continue

    # Extract text
    extracted = extract_article_text(html)
    c["extracted_text_len"] = int(extracted.get("text_len") or 0)

    ogd = (extracted.get("og_description") or "").strip()
    mdd = (extracted.get("meta_description") or "").strip()
    snippet = (extracted.get("text") or "").strip()

    local_best = ogd or mdd
    if not local_best and snippet:
        local_best = snippet[:800].strip()

    # LLM summary
    llm_summary, llm_status = summarize_100w_english(snippet)
    c["llm_status"] = llm_status

    if llm_status == "ok" and llm_summary:
        if write_enriched_into_summary:
            c["summary"] = llm_summary
        else:
            c["summary_enriched"] = llm_summary
        c["summary_source"] = "openai_100w"
        c["confidence"] = 0.85 if c.get("published_known", False) else 0.80
        c["enrich_status"] = "enriched_llm"
        enriched += 1
        continue

    # Fallback: meta/og/snippet
    if local_best:
        if write_enriched_into_summary:
            c["summary"] = local_best
        else:
            c["summary_enriched"] = local_best
        c["summary_source"] = "meta_or_snippet"
        c["confidence"] = 0.70 if (ogd or mdd) else 0.62
        c["enrich_status"] = "enriched_fallback"
        enriched += 1
        continue

    c["enrich_status"] = "no_extractable_text"
    c["confidence"] = float(c.get("confidence") or 0.55)
    skipped += 1
    skip_reasons["no_extractable_text"] += 1

logger.info(f"[{run_id}] Enrichment done: attempted={attempted}, enriched={enriched}, skipped={skipped}")
if skip_reasons:
    logger.info(
        f"[{run_id}] Enrichment skip breakdown: "
        + ", ".join([f"{k}={v}" for k, v in skip_reasons.items()])
    )

logger.info(f"[{run_id}] Cell 05.5 complete: candidates updated with summary/confidence when possible")


2026-01-29 10:18:35 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Starting enrichment (max_enrich_items=1000)
2026-01-29 10:18:35 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Enrichment done: attempted=0, enriched=0, skipped=0
2026-01-29 10:18:35 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Cell 05.5 complete: candidates updated with summary/confidence when possible


In [23]:
# ============================================================
# Cell 06 — Dedup and write Events via 029 (aligned, confidence-aware, date-safe)
# ============================================================
# Overview:
#   Deduplicate event candidates by dedup_key (in-memory + Notion) and create new Events via 029 wrappers:
#     - query_event_by_dedup_key(dedup_key)
#     - create_event(...)
#   Maintain run_stats counters and per-item results (created / skipped / failed).
#
# Inputs / Outputs:
#   Inputs:  event_candidates, run_id, run_stats, config, query_event_by_dedup_key, create_event
#   Outputs: created_events (list), skipped_duplicates (list), failed_events (list)
#
# Notes:
#   - Wrappers only (no raw Notion JSON)
#   - Strict idempotency (safe to rerun)
#   - Notion select values MUST match DB options; defaults are configurable via config
#   - Date is normalized to YYYY-MM-DD; invalid date falls back to detected_at
#

from __future__ import annotations

from datetime import datetime, timezone, date

# ----------------------------
# Small config getter (dot-path)
# ----------------------------
def _cfg_get(path: str, default=None):
    if not isinstance(config, dict):
        return default
    cur = config
    for part in path.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return default
        cur = cur[part]
    return cur

def _to_bool(x, default=False) -> bool:
    if x is None:
        return default
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    if s in {"true", "yes", "1", "on", "enabled"}:
        return True
    if s in {"false", "no", "0", "off", "disabled"}:
        return False
    return default

def _to_float(x, default: float) -> float:
    try:
        return float(x)
    except Exception:
        return default

def _normalize_date_yyyy_mm_dd(x, fallback: str) -> str:
    """
    Normalize input into YYYY-MM-DD string.
      - If already YYYY-MM-DD -> keep
      - If datetime/date -> convert
      - Otherwise -> fallback
    """
    if isinstance(x, date) and not isinstance(x, datetime):
        return x.isoformat()
    if isinstance(x, datetime):
        dt = x if x.tzinfo else x.replace(tzinfo=timezone.utc)
        return dt.date().isoformat()

    s = (str(x).strip() if x is not None else "")
    if not s:
        return fallback

    try:
        _ = date.fromisoformat(s)  # validates YYYY-MM-DD
        return s
    except Exception:
        return fallback

# ----------------------------
# Bind wrappers from 029 (executed via %run -i)
# ----------------------------
query_event_by_dedup_key = globals().get("query_event_by_dedup_key")
create_event = globals().get("create_event")

# ----------------------------
# Defaults for Notion selects (override via config)
# ----------------------------
default_event_type = _cfg_get("monitoring.vc.events_defaults.event_type", "VC")
default_source_select = _cfg_get("monitoring.vc.events_defaults.source", "WEB")
default_status = _cfg_get("monitoring.vc.events_defaults.status", "NEW")
default_action_needed = _to_bool(_cfg_get("monitoring.vc.events_defaults.action_needed", True), default=True)

default_confidence = _to_float(_cfg_get("monitoring.vc.events_defaults.default_confidence", 0.60), 0.60)
skip_if_empty_summary = _to_bool(_cfg_get("monitoring.vc.events_defaults.skip_if_empty_summary", False), default=False)

# ----------------------------
# Safety: wrappers required for writes
# ----------------------------
if not callable(query_event_by_dedup_key) or not callable(create_event):
    logger.error(
        f"[{run_id}] Required 029 wrappers missing. "
        f"query_event_by_dedup_key={callable(query_event_by_dedup_key)}, "
        f"create_event={callable(create_event)}. "
        f"Skipping Notion writes."
    )
    created_events = []
    skipped_duplicates = []
    failed_events = [{"reason": "missing_wrappers", "count": len(event_candidates)}]
else:
    logger.info(f"[{run_id}] Dedup + create Events for {len(event_candidates)} candidate(s)")

    # Ensure counters exist (prevents KeyError)
    run_stats.setdefault("events_created", 0)
    run_stats.setdefault("events_failed", 0)
    run_stats.setdefault("events_skipped_duplicate", 0)

    seen_keys: set[str] = set()
    created_events: list[dict] = []
    skipped_duplicates: list[dict] = []
    failed_events: list[dict] = []

    detected_at = datetime.now(timezone.utc).date().isoformat()

    for i, c in enumerate(event_candidates, start=1):
        title_short = (c.get("title") or "")[:80]
        dedup_key = (c.get("dedup_key") or "").strip()

        try:
            # ----------------------------
            # Basic validation
            # ----------------------------
            if not dedup_key:
                run_stats["events_failed"] += 1
                failed_events.append({"reason": "missing_dedup_key", "title": c.get("title"), "url": c.get("url")})
                continue

            url = (c.get("url") or "").strip()
            if not url:
                run_stats["events_failed"] += 1
                failed_events.append({"reason": "missing_source_url", "dedup_key": dedup_key, "title": c.get("title")})
                continue

            # ----------------------------
            # In-memory dedup
            # ----------------------------
            if dedup_key in seen_keys:
                run_stats["events_skipped_duplicate"] += 1
                skipped_duplicates.append({"dedup_key": dedup_key, "title": c.get("title"), "reason": "in_memory"})
                continue
            seen_keys.add(dedup_key)

            # ----------------------------
            # Notion dedup
            # ----------------------------
            try:
                ex = query_event_by_dedup_key(dedup_key)
            except Exception as e:
                logger.warning(f"[{run_id}] Dedup query failed (will attempt create): {type(e).__name__}: {e}")
                ex = None

            if ex:
                run_stats["events_skipped_duplicate"] += 1
                skipped_duplicates.append({
                    "dedup_key": dedup_key,
                    "title": c.get("title"),
                    "reason": "notion",
                    "page_id": ex.get("page_id"),
                })
                continue

            # ----------------------------
            # Required relation: Target
            # ----------------------------
            target_page_id = c.get("target_page_id") or c.get("source_id")
            if not target_page_id:
                run_stats["events_failed"] += 1
                failed_events.append({
                    "reason": "missing_target_page_id",
                    "dedup_key": dedup_key,
                    "title": c.get("title"),
                    "url": url,
                })
                continue

            # ----------------------------
            # Summary + confidence
            # ----------------------------
            summary = (c.get("summary") or "").strip()
            if skip_if_empty_summary and not summary:
                run_stats["events_failed"] += 1
                failed_events.append({
                    "reason": "empty_summary_skipped",
                    "dedup_key": dedup_key,
                    "title": c.get("title"),
                    "url": url,
                })
                continue

            conf = _to_float(c.get("confidence"), default_confidence)

            # ----------------------------
            # Date handling (critical)
            # ----------------------------
            event_date = _normalize_date_yyyy_mm_dd(c.get("date"), fallback=detected_at)

            if event_date == detected_at and (str(c.get("date") or "").strip() not in {"", detected_at}):
                logger.info(
                    f"[{run_id}] [DATE_WARN] Invalid candidate date -> fallback to detected_at. "
                    f"title='{title_short}' date_in='{c.get('date')}' detected_at='{detected_at}'"
                )

            # ----------------------------
            # Notion select values (safe defaults)
            # ----------------------------
            source_select = str(c.get("source_select") or default_source_select)
            event_type = str(c.get("event_type") or default_event_type)
            status = str(c.get("status") or default_status)
            action_needed = bool(c.get("action_needed", default_action_needed))

            # ----------------------------
            # Create in Notion (via wrapper)
            # ----------------------------
            event = create_event(
                name=c.get("title") or "Untitled",
                date=event_date,
                detected_at=detected_at,
                target_page_ids=[target_page_id],
                event_type=event_type,
                source_url=url,
                source=source_select,
                summary=summary,
                confidence=conf,
                status=status,
                dedup_key=dedup_key,
                run_id=run_id,
                action_needed=action_needed,
                related_paper_ids=None,  # safer than []
            )

            run_stats["events_created"] += 1
            created_events.append(event)

            logger.info(
                f"[{run_id}] ✓ Created Event: {title_short} | date={event_date} | "
                f"conf={conf:.2f} | source_select={source_select}"
            )

        except Exception as e:
            run_stats["events_failed"] += 1
            failed_events.append({
                "dedup_key": dedup_key,
                "title": c.get("title"),
                "url": c.get("url"),
                "reason": f"{type(e).__name__}: {e}",
            })
            logger.warning(f"[{run_id}] ✗ Failed Event create: {title_short} ({type(e).__name__}: {e})")
            continue

    logger.info(
        f"[{run_id}] Cell 06 summary: created={run_stats.get('events_created',0)}, "
        f"dupes={run_stats.get('events_skipped_duplicate',0)}, failed={run_stats.get('events_failed',0)}"
    )

logger.info(f"[{run_id}] Cell 06 complete: Events deduped + written via 029 wrappers")


2026-01-29 10:22:20 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Dedup + create Events for 0 candidate(s)
2026-01-29 10:22:20 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Cell 06 summary: created=0, dupes=3, failed=0
2026-01-29 10:22:20 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Cell 06 complete: Events deduped + written via 029 wrappers


In [24]:
# ============================================================
# Cell 07 — State update and Daily Run Summary (aligned, defensive)
# ============================================================
# Overview:
#   Persist last scan markers + counters via 028 state helpers.
#   Generate a Daily Run Summary in Markdown + Slack-friendly text.
#
# Inputs / Outputs:
#   Inputs:  run_stats, vc_targets, scanner_config,
#            event_candidates, created_events/skipped_duplicates/failed_events
#   Outputs: state updates, daily_summary_md, daily_summary_slack
#
# Notes:
#   - Use update_state() (028)
#   - Be defensive to missing globals
#   - Prefer created_events for "new events"
#

from __future__ import annotations

from datetime import datetime, timezone, timedelta
from collections import defaultdict

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def _safe_int(x, default=0) -> int:
    try:
        return int(x)
    except Exception:
        return default

def _fmt_dt(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%d %H:%M:%S")

JST = timezone(timedelta(hours=9))

# Defensive bindings (avoid NameError)
event_candidates = globals().get("event_candidates") or []
created_events = globals().get("created_events") or []
skipped_duplicates = globals().get("skipped_duplicates") or []
failed_events = globals().get("failed_events") or []

vc_targets = globals().get("vc_targets") or []
scanner_config = globals().get("scanner_config") or {}
run_stats = globals().get("run_stats") or {}

# Ensure expected keys exist
run_stats.setdefault("start_time", datetime.now(timezone.utc).isoformat())
run_stats.setdefault("events_created", 0)
run_stats.setdefault("events_failed", 0)
run_stats.setdefault("events_skipped_duplicate", 0)
run_stats.setdefault("events_skipped_stale", 0)
run_stats.setdefault("targets_processed", 0)
run_stats.setdefault("targets_failed", 0)
run_stats.setdefault("items_fetched", 0)

# ------------------------------------------------------------
# Finalize run stats
# ------------------------------------------------------------
end_dt_utc = datetime.now(timezone.utc)
run_stats["end_time"] = end_dt_utc.isoformat()

try:
    start_dt = datetime.fromisoformat(str(run_stats.get("start_time") or ""))
    if start_dt.tzinfo is None:
        start_dt = start_dt.replace(tzinfo=timezone.utc)
    run_stats["duration_sec"] = round((end_dt_utc - start_dt).total_seconds(), 2)
except Exception as e:
    logger.warning(f"[{run_id}] Could not calculate duration: {e}")
    run_stats["duration_sec"] = 0.0

logger.info(f"[{run_id}] Daily run completed in {run_stats['duration_sec']}s")

# ------------------------------------------------------------
# State updates (028)
# ------------------------------------------------------------
try:
    update_state("last_vc_scan_at", run_stats["end_time"])
    update_state("last_vc_run_stats", run_stats)

    total_created = _safe_int(get_state("vc_total_events_created", 0) or 0, 0)
    total_runs = _safe_int(get_state("vc_total_runs", 0) or 0, 0)

    total_created += _safe_int(run_stats.get("events_created", 0), 0)
    total_runs += 1

    update_state("vc_total_events_created", total_created)
    update_state("vc_total_runs", total_runs)

    logger.info(f"[{run_id}] State updated: last_vc_scan_at, last_vc_run_stats, totals")
except Exception as e:
    logger.warning(f"[{run_id}] State update failed (continuing): {e}")

# ------------------------------------------------------------
# Summary inputs
# ------------------------------------------------------------
freshness_hours = globals().get("freshness_hours")
if freshness_hours is None:
    freshness_hours = (scanner_config or {}).get("freshness_hours", 48)
freshness_hours = _safe_int(freshness_hours, 48)

window_start = (scanner_config or {}).get("window_start")
window_end = (scanner_config or {}).get("window_end")

# Top sources (by candidates)
source_counts = defaultdict(int)
for c in event_candidates:
    source_counts[(c.get("source") or "unknown")] += 1
top_sources_5 = sorted(source_counts.items(), key=lambda x: x[1], reverse=True)[:5]
top_sources_3 = top_sources_5[:3]

end_dt_jst = end_dt_utc.astimezone(JST)

# ------------------------------------------------------------
# Markdown summary
# ------------------------------------------------------------
md = []
md.append("# 📊 VC Monitoring Daily Run Summary")
md.append("")
md.append(f"**Run ID:** `{run_id}`")
md.append(f"**Run Time (UTC):** {_fmt_dt(end_dt_utc)}")
md.append(f"**Run Time (JST):** {_fmt_dt(end_dt_jst)}")
md.append(f"**Duration:** {run_stats.get('duration_sec', 0)}s")
md.append(f"**Window (config):** {window_start} → {window_end}")
md.append(f"**Freshness (rolling):** {freshness_hours}h")
md.append("")
md.append("## 📈 Summary Metrics")
md.append("")
md.append(f"- **Targets Processed:** {run_stats.get('targets_processed', 0)} / {len(vc_targets)}")
md.append(f"- **Targets Failed:** {run_stats.get('targets_failed', 0)}")
md.append(f"- **Items Fetched:** {run_stats.get('items_fetched', 0)}")
md.append(f"- **Event Candidates:** {len(event_candidates)}")
md.append("")
md.append("### ✅ Event Results")
md.append("")
md.append(f"- **Created:** {run_stats.get('events_created', 0)}")
md.append(f"- **Skipped (Duplicate):** {run_stats.get('events_skipped_duplicate', 0)}")
md.append(f"- **Skipped (Stale):** {run_stats.get('events_skipped_stale', 0)}")
md.append(f"- **Failed:** {run_stats.get('events_failed', 0)}")
md.append("")

if top_sources_5:
    md.append("### 🔝 Top Sources (by candidates)")
    md.append("")
    for s, n in top_sources_5:
        md.append(f"- **{s}:** {n}")
    md.append("")

if created_events:
    md.append("### 🆕 New Events Created (up to 5)")
    md.append("")
    for i, ev in enumerate(created_events[:5], start=1):
        title = ev.get("name") or "(no title)"
        date_s = ev.get("date") or ""
        url = ev.get("source_url") or ""
        pid = ev.get("page_id") or ""
        md.append(f"{i}. **{title[:80]}**")
        if date_s:
            md.append(f"   - Date: {date_s}")
        if url:
            md.append(f"   - URL: {url}")
        if pid:
            md.append(f"   - Page ID: `{pid}`")
        md.append("")

if run_stats.get("targets_failed", 0) > 0 or run_stats.get("events_failed", 0) > 0:
    md.append("### ⚠️ Warnings")
    md.append("")
    if run_stats.get("targets_failed", 0) > 0:
        md.append(f"- {run_stats.get('targets_failed', 0)} target(s) failed to fetch/parse")
    if run_stats.get("events_failed", 0) > 0:
        md.append(f"- {run_stats.get('events_failed', 0)} event(s) failed to write to Notion")
    md.append("")
    md.append("_Check logs for details._")
    md.append("")

md.append("---")
md.append("_Generated by 032_monitor_vc_daily.ipynb_")
daily_summary_md = "\n".join(md)

# ------------------------------------------------------------
# Slack summary
# ------------------------------------------------------------
sl = []
sl.append(":bar_chart: *VC Monitoring Daily Run Summary*")
sl.append(f"*Run ID:* `{run_id}`")
sl.append(f":calendar: *Run:* {_fmt_dt(end_dt_jst)} JST  |  {_fmt_dt(end_dt_utc)} UTC")
sl.append(f":stopwatch: *Duration:* {run_stats.get('duration_sec', 0)}s")
sl.append(f":hourglass_flowing_sand: *Window:* {window_start} → {window_end}  |  *{freshness_hours}h rolling*")
sl.append("")
sl.append(f"*Targets:* {run_stats.get('targets_processed',0)}/{len(vc_targets)} processed, {run_stats.get('targets_failed',0)} failed")
sl.append(f"*Items:* {run_stats.get('items_fetched',0)} fetched, {len(event_candidates)} candidates")
sl.append("")
sl.append("*Events:*")
sl.append(f":white_check_mark: Created: {run_stats.get('events_created',0)}")
sl.append(f":fast_forward: Skipped (dup): {run_stats.get('events_skipped_duplicate',0)}")
sl.append(f":zzz: Skipped (stale): {run_stats.get('events_skipped_stale',0)}")
sl.append(f":x: Failed: {run_stats.get('events_failed',0)}")

if top_sources_3:
    sl.append("")
    sl.append("*Top Sources:*")
    for s, n in top_sources_3:
        sl.append(f"• {s}: {n}")

if created_events:
    sl.append("")
    sl.append("*New (sample):*")
    for ev in created_events[:3]:
        title = (ev.get("name") or "")[:80]
        url = (ev.get("source_url") or "")
        sl.append(f"• {title}  —  {url}" if url else f"• {title}")

daily_summary_slack = "\n".join(sl)

# ------------------------------------------------------------
# Persist summaries to state (optional)
# ------------------------------------------------------------
try:
    update_state("last_vc_summary_md", daily_summary_md)
    update_state("last_vc_summary_slack", daily_summary_slack)
except Exception as e:
    logger.warning(f"[{run_id}] Could not store summaries in state: {e}")

# Preview (logs)
logger.info("\n" + "=" * 60)
logger.info("DAILY RUN SUMMARY (Slack)")
logger.info("=" * 60)
logger.info(daily_summary_slack)
logger.info("=" * 60)

logger.info("\n" + "=" * 60)
logger.info("DAILY RUN SUMMARY (Markdown)")
logger.info("=" * 60)
logger.info(daily_summary_md)
logger.info("=" * 60)

logger.info(f"[{run_id}] Cell 07 complete: state updated + summaries generated")


2026-01-29 10:23:58 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Daily run completed in 0.0s
2026-01-29 10:23:58 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] State updated: last_vc_scan_at, last_vc_run_stats, totals
2026-01-29 10:23:58 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | 
2026-01-29 10:23:58 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | DAILY RUN SUMMARY (Slack)
2026-01-29 10:23:58 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | ============================================================
2026-01-29 10:23:58 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | :bar_chart: *VC Monitoring Daily Run Summary*
*Run ID:* `8fc0a521-19f9-49be-a7d9-2584732c4be7`
:calendar: *Run:* 2026-01-29 10:23:58 JST  |  2026-01-29 01:23:58 UTC
:stopwatch: *Duration:* 0.0s
:hourglass_flowing_sand: *Window:* 2026-01-22 → 2026-01-29  |  *48h rolling*

*Targets:* 9/9 processed, 0 failed
*Items:* 

In [25]:
# ============================================================
# Cell 08 — Optional local summary save (aligned)
# ============================================================
# Overview:
#   Optionally save daily run summary to local files for archival/review.
#   Write both Markdown and Slack formats to timestamped files.
#
# Inputs / Outputs:
#   Inputs:  daily_summary_md, daily_summary_slack, run_stats, config
#   Outputs: Local .md and .txt files (optional), state paths updated (optional)
#
# Notes:
#   - Optional feature; failures logged but don't break workflow
#   - Read config defensively (no get_config)
#   - Use update_state (no set_state)
#

from pathlib import Path
from datetime import datetime, timezone

def _cfg_get(path: str, default=None):
    if not isinstance(config, dict):
        return default
    cur = config
    for part in path.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return default
        cur = cur[part]
    return cur

def _to_bool(x, default=False) -> bool:
    if x is None:
        return default
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    if s in {"true", "yes", "1", "on", "enabled"}:
        return True
    if s in {"false", "no", "0", "off", "disabled"}:
        return False
    return default

def _to_int(x, default: int) -> int:
    try:
        return int(x)
    except Exception:
        return default

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
save_local_summary = _to_bool(_cfg_get("monitoring.vc.save_local_summary", False), default=False)
summary_dir_str = _cfg_get("monitoring.vc.summary_dir", "./summaries")
max_summary_files = _to_int(_cfg_get("monitoring.vc.max_summary_files", 30), default=30)

if not save_local_summary:
    logger.info(f"[{run_id}] Local summary save disabled (monitoring.vc.save_local_summary=False)")
else:
    try:
        logger.info(f"[{run_id}] Saving daily run summaries locally...")

        summary_dir = Path(summary_dir_str)
        summary_dir.mkdir(parents=True, exist_ok=True)

        # Timestamp for filename
        timestamp_str = None
        try:
            end_time = run_stats.get("end_time")
            if end_time:
                end_dt = datetime.fromisoformat(str(end_time))
                if end_dt.tzinfo is None:
                    end_dt = end_dt.replace(tzinfo=timezone.utc)
                timestamp_str = end_dt.strftime("%Y%m%d_%H%M%S")
        except Exception:
            timestamp_str = None

        if not timestamp_str:
            timestamp_str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

        md_path = summary_dir / f"vc_summary_{timestamp_str}.md"
        slack_path = summary_dir / f"vc_summary_{timestamp_str}.txt"

        # Write files
        try:
            md_path.write_text(daily_summary_md or "", encoding="utf-8")
            logger.info(f"[{run_id}] ✓ Markdown summary saved: {md_path}")
        except Exception as e:
            logger.warning(f"[{run_id}] Failed to write Markdown summary: {e}")

        try:
            slack_path.write_text(daily_summary_slack or "", encoding="utf-8")
            logger.info(f"[{run_id}] ✓ Slack summary saved: {slack_path}")
        except Exception as e:
            logger.warning(f"[{run_id}] Failed to write Slack summary: {e}")

        # Retention cleanup (best-effort)
        try:
            md_files = sorted(summary_dir.glob("vc_summary_*.md"), key=lambda p: p.stat().st_mtime, reverse=True)
            if len(md_files) > max_summary_files:
                for old_md in md_files[max_summary_files:]:
                    try:
                        old_txt = old_md.with_suffix(".txt")
                        old_md.unlink(missing_ok=True)
                        if old_txt.exists():
                            old_txt.unlink()
                    except Exception as e:
                        logger.debug(f"[{run_id}] Could not delete old summary {old_md.name}: {e}")
                logger.info(f"[{run_id}] Retention cleanup complete (keep={max_summary_files})")
        except Exception as e:
            logger.debug(f"[{run_id}] Retention cleanup skipped: {e}")

        # Store paths in state (optional)
        try:
            update_state("last_vc_summary_md_path", str(md_path))
            update_state("last_vc_summary_slack_path", str(slack_path))
        except Exception as e:
            logger.debug(f"[{run_id}] Could not store summary paths in state: {e}")

    except Exception as e:
        logger.warning(f"[{run_id}] Local summary save failed (non-critical): {e}")

logger.info(f"[{run_id}] Cell 08 complete: optional local summary save processed")


2026-01-29 10:24:10 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Local summary save disabled (monitoring.vc.save_local_summary=False)
2026-01-29 10:24:10 | INFO     | 8fc0a521-19f9-49be-a7d9-2584732c4be7 | [8fc0a521-19f9-49be-a7d9-2584732c4be7] Cell 08 complete: optional local summary save processed
